# DICE Analysis and Results (Portable)

This notebook is the portable replacement for the original Mac-specific notebook. It resolves the repository paths dynamically, uses the same wrapper as the terminal flow, and can be run from a Linux server, macOS, or Windows as long as the released dataset is present in the clone.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

import pandas as pd
from IPython.display import Image, display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [ ]:
def resolve_analysis_root(start: Path) -> tuple[Path, Path]:
    for base in [start, *start.parents]:
        if base.name == 'analysis and results' and (base / 'tools' / 'run_results_pipeline.py').exists():
            return base, base.parent
        candidate = base / 'analysis and results'
        if (candidate / 'tools' / 'run_results_pipeline.py').exists():
            return candidate, candidate.parent
    raise RuntimeError('Could not locate the analysis and results folder from the current working directory.')

def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env

ANALYSIS_ROOT, REPO_ROOT = resolve_analysis_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
WRAPPER = ANALYSIS_ROOT / 'tools' / 'run_results_pipeline.py'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_FULL_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_TUNE = DATASET_ROOT / 'results_dice_tuning'

print('ANALYSIS_ROOT:', ANALYSIS_ROOT)
print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)
print('WRAPPER      :', WRAPPER)


## Run Base Pipeline

This runs the same wrapper used by the terminal workflow and produces `results_analysis/` and `results_dice_full/`.


In [ ]:
RUN_PIPELINE = True

if RUN_PIPELINE:
    cmd = [sys.executable, str(WRAPPER)]
    proc = subprocess.run(
        cmd,
        cwd=str(ANALYSIS_ROOT),
        env=portable_env(),
        capture_output=True,
        text=True,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f'run_results_pipeline.py failed with exit code {proc.returncode}')
else:
    print('Skipped base pipeline run. Set RUN_PIPELINE=True to execute.')


In [ ]:
overall = pd.read_csv(OUT / 'table_overall_metrics.csv')
stressor = pd.read_csv(OUT / 'table_stressor_metrics.csv')
workload = pd.read_csv(OUT / 'table_workload_summary.csv')
features = pd.read_csv(OUT / 'table_feature_inventory.csv')
quality = pd.read_csv(OUT / 'table_case_quality.csv')

print('Overall metrics')
display(overall)

print('Per-stressor metrics')
display(stressor)

print('Workload summary')
display(workload)

print('Feature inventory')
display(features[['tier_name', 'n_features_common', 'n_features_union']])

print('Case quality snapshot')
display(quality.head())


In [ ]:
figure_paths = [
    FIG / 'fig_af_timeseries_tier0.png',
    FIG / 'fig_af_timeseries_tier1_alt.png',
    FIG / 'fig_af_timeseries_tier2.png',
    FIG / 'fig_heatmap_roc_auc.png',
    FIG / 'fig_heatmap_pr_auc.png',
    FIG / 'fig_run_score_distributions.png',
]

for path in figure_paths:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('Missing:', path)

print('LaTeX table (overall) :', OUT / 'table_overall_metrics.tex')
print('LaTeX table (stressor):', OUT / 'table_stressor_metrics.tex')
print('Markdown summary      :', OUT / 'RESULTS_SUMMARY.md')


## Full DICE Retrain + Conformal Pipeline

The base wrapper run already generates `results_dice_full/`. The cells below load those outputs and optionally run holdout or tuning modes.


In [ ]:
overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
stressor_full = pd.read_csv(OUT_FULL / 'stressor_metrics_final_config.csv')
fold_full = pd.read_csv(OUT_FULL / 'fold_metrics.csv')
case_full = pd.read_csv(OUT_FULL / 'case_predictions.csv')

print('Overall full-pipeline metrics')
display(overall_full)

print('Final config stressor metrics')
display(stressor_full)

print('Fold metrics')
display(fold_full)

row_final = overall_full[overall_full['config'] == 'tier0_tier1_tier2'].iloc[0]
print('Final config (tier0_tier1_tier2):')
print('Base ROC-AUC :', round(float(row_final['roc_auc']), 4))
print('Base AUC-PR  :', round(float(row_final['pr_auc']), 4))
print('WC ROC-AUC   :', round(float(row_final['roc_auc_wc']), 4))
print('WC AUC-PR    :', round(float(row_final['pr_auc_wc']), 4))
print('Run-FPR      :', round(float(row_final['fpr_run_alert']), 4))
print('Run-TPR      :', round(float(row_final['tpr_run_alert']), 4))


In [ ]:
RUN_HOLDOUT = False

if RUN_HOLDOUT:
    cmd = [sys.executable, str(WRAPPER), "--skip_analysis", "--skip_full", "--run_holdout"]
    proc = subprocess.run(
        cmd,
        cwd=str(ANALYSIS_ROOT),
        env=portable_env(),
        capture_output=True,
        text=True,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f'holdout run failed with exit code {proc.returncode}')
else:
    print('Skipped holdout run. Set RUN_HOLDOUT=True to execute.')


In [ ]:
RUN_TUNING = False

if RUN_TUNING:
    cmd = [sys.executable, str(WRAPPER), "--skip_analysis", "--skip_full", "--run_tuning"]
    proc = subprocess.run(
        cmd,
        cwd=str(ANALYSIS_ROOT),
        env=portable_env(),
        capture_output=True,
        text=True,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f'tuning run failed with exit code {proc.returncode}')
else:
    print('Skipped tuning run. Set RUN_TUNING=True to execute.')


In [ ]:
if (OUT_TUNE / "recommended_config.json").exists():
    s1 = pd.read_csv(OUT_TUNE / 'sweep_stage1_gain_block.csv')
    s2 = pd.read_csv(OUT_TUNE / 'sweep_stage2_alpha_persist.csv')
    rec_cfg = json.loads((OUT_TUNE / 'recommended_config.json').read_text())
    rec_overall = pd.read_csv(OUT_TUNE / 'recommended_overall_metrics.csv')
    rec_stressor = pd.read_csv(OUT_TUNE / 'recommended_stressor_metrics.csv')
    rec_fold = pd.read_csv(OUT_TUNE / 'recommended_fold_metrics.csv')

    print('Recommended config:', rec_cfg)

    print('Top Stage-1 (gain/block_B by base ROC):')
    display(s1.sort_values(['roc_auc', 'pr_auc', 'tpr_run_alert'], ascending=False).head(10))

    feasible = s2[s2['fpr_run_alert'] <= 0.25]
    if feasible.empty:
        feasible = s2
    print('Top Stage-2 (alpha/persist):')
    display(feasible.sort_values(['tpr_run_alert', 'roc_auc', 'pr_auc'], ascending=False).head(10))

    print('Recommended overall metrics:')
    display(rec_overall)

    print('Recommended stressor metrics:')
    display(rec_stressor)

    print('Recommended fold metrics:')
    display(rec_fold)
else:
    print('No tuning outputs yet. Set RUN_TUNING=True in the previous cell to generate them.')
